In [2]:
import pandas as pd
import numpy as np
import json
import glob
import csv
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models

In [3]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [4]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [5]:
# Load the JSON data from the file
with open("vir_transcripts.json", "r") as file:
    data = json.load(file)

# Get the keys of the JSON object
fields = data.keys()

# Print the fields
print(fields)


dict_keys(['Transcripts'])


In [6]:
data = load_data("vir_transcripts.json")["Transcripts"]
print (data[0][0:90])
print (data[1][0:90])

I lost 80% of my mind. It is very freeing. You should see the look on your faces right now
Yeah, that was a great transition. I went from an English medium school to a school where 


In [7]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:150])

lose % mind very freeing see look face right now way good evening guy excite all right name ’re go good time tonight ’m so excited ’ go delightful tal


In [8]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [27]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'get', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [28]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
print (data_words[1][0:20])

['lose', 'mind', 'very', 'freeing', 'see', 'look', 'face', 'right', 'now', 'way', 'good', 'evening', 'guy', 'excite', 'all', 'right', 'name', 're', 'go', 'good']
['great', 'transition', 'go', 'english', 'medium', 'school', 'school', 'speak', 'medium', 'go', 'noun', 'pronoun', 'verb', 'chest', 'shoulder', 'tricep', 'go', 'examination', 'thing', 'here']


In [29]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['lose', 'mind', 'freeing', 'look', 'face', 'right', 'way', 'good', 'evening', 'excite', 'right', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'make', 'comedy', 'authentically', 'indian', 'really', 'indian', 'fake', 'american']


In [30]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['lose', 'mind', 'freeing', 'look', 'face', 'right', 'way', 'good', 'evening', 'excite', 'right', 'name', 'good', 'time', 'tonight', 'excited', 'delightful', 'talk', 'time', 'really', 'embrace', 'root', 'make', 'comedy', 'authentically', 'indian', 'really', 'indian', 'fake', 'american', 'accent', 'understand', 'opportunity', 'make', 'history', 'tonight', 'first', 'ever', 'indian', 'come', 'leave', 'never', 'happen', 'stick', 'around', 'kick', 'news', 'week', 'work', 'well', 'leave', 'browner', 'pasture', 'honestly', 'honestly', 'government', 'ban', 'beef', 'international', 'career', 'bad', 'thing', 'make', 'mistake', 'beef', 'good', 'couple', 'first', 'world', 'tour', 'entire', 'world', 'like', 'country', 'world', 'common', 'like', 'thing', 'number', 'masturbate', 'country', 'thank', 'chain', 'dna', 'everywhere', 'hotel', 'memory', 'foam', 'mattress', 'memory', 'matter', 'entire', 'world', 'people', 'thing', 'indian', 'love', 'indian', 'people', 'smart', 'indian', 'people', 'smart', 'l

In [31]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 2), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 2), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1)]


In [37]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=5,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [39]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=5)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.152475  0.094666       1        1  32.436738
2      0.074759 -0.130691       2        1  24.690477
1     -0.131001 -0.069218       3        1  21.462739
4     -0.082030  0.118842       4        1  21.409268
3     -0.014203 -0.013599       5        1   0.000777, topic_info=           Term       Freq      Total Category  logprob  loglift
299       woman  13.000000  13.000000  Default   5.0000   5.0000
597         lie  10.000000  10.000000  Default   4.0000   4.0000
697        want  11.000000  11.000000  Default   3.0000   3.0000
169         man  24.000000  24.000000  Default   2.0000   2.0000
140      indian  17.000000  17.000000  Default   1.0000   1.0000
299       woman  12.828661  13.282321   Topic1  -3.8502   1.0911
697        want  11.349535  11.794040   Topic1  -3.9727   1.0875
69        dream   9.099482   9.539868   Topic1  -4.1937   1.0786
728        ball   7.618680   8.045660   Topic1  -4.3713   1.0713
296        well   6.875688   7.297516   Topic1  -4.4739   1.0663
169         man  19.596289  24.108875   Topic1  -3.4266   0.9186
245        shit  14.347660  30.255005   Topic1  -3.7383   0.3798
140      indian  10.619921  17.678260   Topic1  -4.0392   0.6163
597         lie  10.512558  10.969920   Topic2  -3.7765   1.3562
353        cool   4.982382   5.418072   Topic2  -4.5231   1.3149
581        hurt   3.597376   4.026736   Topic2  -4.8488   1.2860
340       child   2.906554   3.332957   Topic2  -5.0621   1.2619
610        next   2.904135   3.330580   Topic2  -5.0629   1.2617
38         call   2.905304   3.332098   Topic2  -5.0625   1.2617
24    beautiful   9.143702  14.658544   Topic2  -3.9160   0.9268
270        tell   7.753665  16.257035   Topic2  -4.0809   0.6584
161        look   7.756632  18.243664   Topic2  -4.0805   0.5435
27      believe   7.062830  15.217445   Topic2  -4.1742   0.6312
459    religion   7.384538   7.840751   Topic3  -3.9896   1.4789
504       write   5.413227   5.858787   Topic3  -4.3001   1.4598
385      forest   4.089253   4.528565   Topic3  -4.5806   1.4368
369         eat   4.089745   4.530875   Topic3  -4.5805   1.4364
92         fact   3.429283   3.865720   Topic3  -4.7566   1.4191
123       great   3.429851   3.866639   Topic3  -4.7564   1.4190
424      letter   3.432668   3.871272   Topic3  -4.7556   1.4186
245        shit   7.435461  30.255005   Topic3  -3.9827   0.1355
47         come   4.788746  17.579226   Topic3  -4.4227   0.2384
110        fuck   4.107727  13.325193   Topic3  -4.5761   0.3621
251       smart   5.418903   5.866929   Topic4  -4.2966   1.4619
39          car   4.762189   5.203296   Topic4  -4.4257   1.4528
216        poor   4.760526   5.201575   Topic4  -4.4261   1.4527
213     percent   3.437241   3.872699   Topic4  -4.7518   1.4221
54      country   3.436671   3.872637   Topic4  -4.7519   1.4219
140      indian   6.740009  17.678260   Topic4  -4.0784   0.5771
303       world   6.090674  15.521629   Topic4  -4.1797   0.6059
167        make   5.429328  15.782827   Topic4  -4.2946   0.4742
212      people   4.769969  14.551332   Topic4  -4.4241   0.4260
1    absolutely   0.000014   0.555250   Topic5  -6.9379   1.1782
11       answer   0.000014   0.555250   Topic5  -6.9379   1.1782
13        apply   0.000014   0.555250   Topic5  -6.9379   1.1782
14        armed   0.000014   0.555250   Topic5  -6.9379   1.1782
15       around   0.000014   0.555250   Topic5  -6.9379   1.1782
245        shit   0.000020  30.255005   Topic5  -6.5612  -2.4431
459    religion   0.000018   7.840751   Topic5  -6.6939  -1.2254
121        good   0.000019  15.216476   Topic5  -6.6365  -1.8311
303       world   0.000019  15.521629   Topic5  -6.6451  -1.8596
483       story   0.000018  14.572505   Topic5  -6.6813  -1.8327
232       right   0.000019  21.779353   Topic5  -6.6535  -2.2067, token_table=      Topic      Freq       Term
term                   

In [18]:
# import matplotlib.pyplot as plt

# def visualize_lda_topics(lda_model, num_words=10):
#     topics = lda_model.show_topics(num_topics=-1, num_words=num_words, formatted=False)
    
#     for topic_id, words in topics:
#         word_list = [word[0] for word in words]
#         word_probs = [word[1] for word in words]
        
#         plt.figure(figsize=(8, 6))
#         plt.barh(word_list, word_probs, color='skyblue')
#         plt.xlabel('Word Probability')
#         plt.ylabel('Word')
#         plt.title(f'Topic {topic_id + 1}')
#         plt.gca().invert_yaxis()
#         plt.show()

# # Usage
# visualize_lda_topics(lda_model)


In [25]:
!pip show pyLDAvis

Name: pyLDAvis
Version: 3.4.1
Summary: Interactive topic model visualization. Port of the R package.
Home-page: https://github.com/bmabey/pyLDAvis
Author: Ben Mabey
Author-email: ben@benmabey.com
License: BSD-3-Clause
Location: /Users/alisha/anaconda3/envs/data_story/lib/python3.11/site-packages
Requires: funcy, gensim, jinja2, joblib, numexpr, numpy, pandas, scikit-learn, scipy, setuptools
Required-by: 


In [7]:
import sys
print(sys.version)


3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]


In [14]:
!python --version


Python 3.11.5


In [15]:
import pyLDAvis
import pyLDAvis.gensim
vis = pyLDAvis.gensim.prepare(topic_model=lda_model, 
                              corpus=bow2doc_corpus, 
                              dictionary=dictionary)
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)

NameError: name 'bow2doc_corpus' is not defined

In [2]:
!python --version


Python 3.11.5


In [16]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis

/Users/alisha/anaconda3/envs/comedy/lib/python3.11/site-packages/pyLDAvis/_prepare.py:9: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Users/alisha/anaconda3/envs/comedy/lib/python3.11/site-packages/pyLDAvis/_prepare.py:9: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pa

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
18    -0.263323 -0.266651       1        1  32.374234
0      0.021559 -0.302398       2        1  24.588748
24    -0.304909  0.009118       3        1  21.327236
12     0.248342 -0.208836       4        1  21.063683
5      0.045475 -0.025267       5        1   0.534882
28     0.009713  0.030382       6        1   0.004797
11     0.010124  0.031343       7        1   0.004792
6      0.009958  0.031287       8        1   0.004758
19     0.010241  0.032133       9        1   0.004752
13     0.009791  0.031780      10        1   0.004569
9      0.010092  0.031463      11        1   0.004559
15     0.010273  0.031442      12        1   0.004556
22     0.010401  0.031394      13        1   0.004555
23     0.010279  0.031496      14        1   0.004552
4      0.010056  0.031811      15        1   0.004551
29     0.009763  0.031019      16        1   0.004549
7      0.010355  0.031758      17        1   0.004541
25     0.010253  0.031963      18        1   0.004534
27     0.010203  0.032069      19        1   0.004529
8      0.009982  0.031754      20        1   0.004528
21     0.010301  0.032217      21        1   0.004507
14     0.009754  0.031578      22        1   0.004492
2      0.010151  0.032073      23        1   0.004466
20     0.009710  0.031835      24        1   0.004259
3      0.010116  0.032174      25        1   0.004234
16     0.010278  0.032089      26        1   0.004231
17     0.010227  0.032207      27        1   0.004230
10     0.010275  0.032278      28        1   0.004206
1      0.010267  0.032140      29        1   0.004169
26     0.010291  0.032347      30        1   0.003302, topic_info=          Term       Freq      Total Category  logprob  loglift
170        man  30.000000  30.000000  Default  30.0000  30.0000
141     indian  22.000000  22.000000  Default  29.0000  29.0000
300      woman  16.000000  16.000000  Default  28.0000  28.0000
246       shit  39.000000  39.000000  Default  27.0000  27.0000
117        get  39.000000  39.000000  Default  26.0000  26.0000
..         ...        ...        ...      ...      ...      ...
23       basis   0.000062   1.056303  Topic30  -6.9177   0.5777
24   beautiful   0.000062  18.112646  Topic30  -6.9177  -2.2642
25     bedroom   0.000062   3.824891  Topic30  -6.9177  -0.7091
26        beef   0.000062  16.119536  Topic30  -6.9177  -2.1476
27     believe   0.000062  19.709913  Topic30  -6.9177  -2.3487

[1846 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
0         3  0.946157      abki
2         3  0.946330    absorb
3         3  0.946666    accent
4         3  1.010003    accept
511       2  1.024892  actually
...     ...       ...       ...
304       1  0.146286     world
304       2  0.195047     world
304       3  0.390095     world
304       4  0.292571     world
505       4  0.928925     write

[310 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[19, 1, 25, 13, 6, 29, 12, 7, 20, 14, 10, 16, 23, 24, 5, 30, 8, 26, 28, 9, 22, 15, 3, 21, 4, 17, 18, 11, 2, 27])

In [36]:
import pyLDAvis.gensim_models as gensimvis
from gensim import corpora, models, similarities
dictionary = corpora.Dictionary(corpus)

pyLDAvis.enable_notebook()
vis = gensimvis.prepare(lda_model, corpus, dictionary)
vis

TypeError: decoding to str: need a bytes-like object, tuple found

In [ ]:
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis